In [ ]:
# ===============================
# 2. 모델링 & 평가 (저장된 파일 사용)
# ===============================
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV, KFold, cross_val_predict, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.under_sampling import RandomUnderSampler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import warnings
warnings.filterwarnings("ignore")

# 2-1. 전처리된 파일 불러오기
X = pd.read_csv("X_preprocessed.csv")
y = pd.read_csv("y_preprocessed.csv").squeeze()  # Series 형태로 변환

# -------------------------------
# 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba, fold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Fold": fold,
        "Algorithm": name,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

# -------------------------------
# 파라미터 그리드 & 모델 정의
param_grids = {
    "Logistic Regression": {'C': [0.1, 1, 10]},
    "Random Forest": {'n_estimators': [100, 200], 'max_depth': [5, 10]},
    "XGBoost": {'n_estimators': [100], 'max_depth': [3, 5]},
    "LightGBM": {'n_estimators': [100], 'num_leaves': [31, 50]},
    "CatBoost": {'depth': [4, 6], 'iterations': [100], 'learning_rate': [0.1]},
    "MLP": {'hidden_layer_sizes': [(64,), (64, 32)], 'alpha': [0.0001, 0.001]}
}

base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(max_iter=300, random_state=42)
}

# -------------------------------
# 교차검증 학습 + 평가
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)

    rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
    X_res, y_res = rus.fit_resample(X_train_scaled, y_train)

    for name, model in base_models.items():
        clf = GridSearchCV(model, param_grids[name], cv=3, scoring='f1', n_jobs=-1)
        clf.fit(X_res, y_res)
        best_model = clf.best_estimator_
        y_pred = best_model.predict(X_valid_scaled)
        y_proba = best_model.predict_proba(X_valid_scaled)[:, 1]
        results.append(evaluate_model(name, y_valid, y_pred, y_proba, fold))

    # CNN
    cnn = Sequential([
        Dense(64, activation='relu', input_shape=(X_res.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_res, y_res, epochs=10, batch_size=32, verbose=0)
    y_pred_cnn = cnn.predict(X_valid_scaled).flatten()
    y_pred_cnn_label = (y_pred_cnn >= 0.5).astype(int)
    results.append(evaluate_model("CNN", y_valid, y_pred_cnn_label, y_pred_cnn, fold))

# -------------------------------
# KNN 평가
X_scaled_all = MinMaxScaler().fit_transform(X)
X_res_all, y_res_all = RandomUnderSampler(sampling_strategy=0.5, random_state=42).fit_resample(X_scaled_all, y)

knn = KNeighborsClassifier(n_neighbors=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict')
y_proba_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict_proba')[:, 1]
results.append(evaluate_model("KNN", y_res_all, y_pred_knn, y_proba_knn, fold='All'))

# -------------------------------
# 결과 정리
result_df = pd.DataFrame(results)

summary = result_df.groupby("Algorithm")[[ 
    "Precision", "Recall", "Specificity", "F1-score", "Accuracy", "Balanced Accuracy", "AUC"
]].mean().round(4).sort_values("AUC", ascending=False)

print(" 전체 Fold 결과:")
display(result_df)

print("\n 모델별 평균 성능 요약:")
display(summary)


In [ ]:
## 빠른 실행

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from imblearn.under_sampling import RandomUnderSampler

# -------------------------------
# 1. 전처리된 데이터 불러오기
X = pd.read_csv("X_preprocessed.csv")
y = pd.read_csv("y_preprocessed.csv").squeeze()

# -------------------------------
# 2. 평가 함수
def evaluate_model(y_true, y_pred, y_proba, fold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Fold": fold,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

# -------------------------------
# 3. StratifiedKFold + Logistic Regression
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    # 스케일링
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)

    # 언더샘플링
    rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
    X_res, y_res = rus.fit_resample(X_train_scaled, y_train)

    # 로지스틱 회귀 모델 학습
    model = LogisticRegression(max_iter=1000)
    model.fit(X_res, y_res)

    y_pred = model.predict(X_valid_scaled)
    y_proba = model.predict_proba(X_valid_scaled)[:, 1]

    results.append(evaluate_model(y_valid, y_pred, y_proba, fold))

# -------------------------------
# 4. 결과 출력
result_df = pd.DataFrame(results)
print("Fold별 성능:")
print(result_df)

print("\n평균 성능:")
print(result_df.mean(numeric_only=True).round(4))
